# ハッブル・テンション: 方法ごとの $H_0$ を並べて比べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coxless/astro-sandbox/blob/main/cosmology/hubble_tension.ipynb)

[hz_comparison.ipynb](hz_comparison.ipynb) で、宇宙の膨張の歴史の **形** はどの方法でもそろうのに、**今の膨張の速さ $H_0$** は
「距離はしご」を使うと約 73、「初期宇宙の物差し」を使うと約 67〜69 に分かれることを見ました。これが **ハッブル・テンション** です。

このノートブックでは

1. 論文で発表された $H_0$ を方法ごとに並べ、どれくらい食い違っているか(何σか)を計算する
2. CMB を使わずに、BAO と **ビッグバン元素合成(BBN)** だけから $H_0$ を自分で出す
3. 「どこが何 % ずれていれば食い違いが消えるか」を、物差し $r_d$ と、[distance/cepheid.ipynb](../distance/cepheid.ipynb) で求めた LMC までの距離について計算する
4. 近くの宇宙だけ特別に速く膨張している(局所的な「泡」)せいではないか、超新星を赤方偏移で分けて確かめる

## 使うデータ

- 各 $H_0$ の値: 下の表に出典の論文を書いています(値は論文の要旨から)
- DESI DR2 BAO(DESI Collaboration 2025): https://github.com/CobayaSampler/bao_data
- Pantheon+(Scolnic et al. 2022, Brout et al. 2022): https://github.com/PantheonPlusSH0ES/DataRelease

データのリポジトリにはライセンスの明記はありません。研究用に公開されたデータなので、使うときは論文を引用します。

## 0. 準備

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BAO_URL = "https://raw.githubusercontent.com/CobayaSampler/bao_data/master/desi_bao_dr2/"
SN_DATA_URL = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/"
    "Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES.dat"
)
C_KMS = 299_792.458  # 光速 [km/s]

## 1. 発表されている $H_0$ を集める

大きく 2 つのグループに分けます。

- **初期宇宙(early)**: 宇宙が生まれて 38 万年ごろの物理(CMB、音響地平線 $r_d$、元素合成)を物差しにして、ΛCDM で今まで外挿する
- **近傍宇宙(late)**: 今の宇宙で直接距離を測る(距離はしご、重力レンズの時間の遅れ、重力波など)

誤差が上下で違うものは `err_lo`, `err_hi` に分けて入れています。統計誤差と系統誤差が別々に書かれているものは、2 乗和の平方根でまとめました。

In [ ]:
H0_TABLE = pd.DataFrame(
    [
        # name, group, method, H0, err_lo, err_hi, reference
        ("Planck 2018", "early", "CMB", 67.36, 0.54, 0.54, "Planck Collaboration (2020)"),
        ("ACT DR4", "early", "CMB", 67.9, 1.5, 1.5, "Aiola et al. (2020)"),
        ("SPT-3G 2018", "early", "CMB", 68.3, 1.5, 1.5, "Balkenhol et al. (2023)"),
        ("DESI DR2 BAO + BBN", "early", "BAO", 68.51, 0.58, 0.58, "DESI Collaboration (2025)"),
        ("SH0ES (Cepheid + SN Ia)", "late", "distance ladder", 73.04, 1.04, 1.04, "Riess et al. (2022)"),
        ("CCHP (TRGB + SN Ia, 2021)", "late", "distance ladder", 69.8, 1.7, 1.7, "Freedman (2021)"),
        ("CCHP (TRGB + SN Ia, HST+JWST)", "late", "distance ladder", 70.39, 1.94, 1.94, "Freedman et al. (2025)"),
        ("SBF", "late", "distance ladder", 73.3, 2.5, 2.5, "Blakeslee et al. (2021)"),
        ("Megamasers (MCP)", "late", "geometric", 73.9, 3.0, 3.0, "Pesce et al. (2020)"),
        ("H0LiCOW lensing", "late", "time delay", 73.3, 1.8, 1.7, "Wong et al. (2020)"),
        ("TDCOSMO 2025 lensing", "late", "time delay", 71.6, 3.3, 3.9, "TDCOSMO Collaboration (2025)"),
        ("GW170817 standard siren", "late", "gravitational wave", 70.0, 8.0, 12.0, "Abbott et al. (2017)"),
    ],
    columns=["name", "group", "method", "H0", "err_lo", "err_hi", "reference"],
)
H0_TABLE["err"] = (H0_TABLE["err_lo"] + H0_TABLE["err_hi"]) / 2  # 計算用に左右の平均
H0_TABLE

前のノートブックで自分で出した値も加えておきます(論文の値ではないので、図では白抜きにします)。

In [ ]:
MINE = pd.DataFrame(
    [
        ("this repo: Pantheon+ × Cepheid M_B", "late", "distance ladder", 73.4, 1.0, 1.0, "sn_cosmology_fit.ipynb"),
        ("this repo: cosmic chronometers", "late", "chronometer", 65.9, 5.7, 5.7, "hz_comparison.ipynb"),
        ("this repo: DESI BAO + rd(Planck)", "early", "BAO", 69.0, 0.5, 0.5, "hz_comparison.ipynb"),
    ],
    columns=["name", "group", "method", "H0", "err_lo", "err_hi", "reference"],
)
MINE["err"] = (MINE["err_lo"] + MINE["err_hi"]) / 2

In [ ]:
def whisker_plot(table, mine=None, bands=()):
    rows = pd.concat([table.assign(mine=False), mine.assign(mine=True)] if mine is not None else [table.assign(mine=False)])
    rows = rows.sort_values(["group", "H0"], ascending=[True, True]).reset_index(drop=True)
    colors = {"early": "tab:blue", "late": "tab:red"}
    fig, ax = plt.subplots(figsize=(8, 0.42 * len(rows) + 1))
    for y, r in rows.iterrows():
        ax.errorbar(r["H0"], y, xerr=[[r["err_lo"]], [r["err_hi"]]], fmt="o", c=colors[r["group"]],
                    mfc="white" if r["mine"] else colors[r["group"]], capsize=3)
    for center, width, color in bands:
        ax.axvspan(center - width, center + width, color=color, alpha=0.15)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels(rows["name"])
    ax.set_xlabel("H0 [km/s/Mpc]")
    ax.set_xlim(60, 85)
    ax.plot([], [], "o", c="tab:blue", label="early universe")
    ax.plot([], [], "o", c="tab:red", label="late universe")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


planck = H0_TABLE.set_index("name").loc["Planck 2018"]
shoes = H0_TABLE.set_index("name").loc["SH0ES (Cepheid + SN Ia)"]
whisker_plot(H0_TABLE, MINE, bands=[(planck["H0"], planck["err"], "tab:blue"), (shoes["H0"], shoes["err"], "tab:red")])

## 2. 食い違いは何σか

2 つの独立な測定 $H_1 \pm \sigma_1$, $H_2 \pm \sigma_2$ の差を、その誤差で割ったものを「何σの食い違いか」の目安にします。

$$
n_\sigma = \frac{|H_1 - H_2|}{\sqrt{\sigma_1^2 + \sigma_2^2}}
$$

(誤差が正規分布という仮定です。上下で誤差が違うものは平均で近似しています。)

In [ ]:
def tension(a, b):
    return abs(a["H0"] - b["H0"]) / np.hypot(a["err"], b["err"])


t = H0_TABLE.set_index("name")
for name in t.index:
    if name == "Planck 2018":
        continue
    print(f"{name:32s} {t.loc[name, 'H0']:6.2f}   vs Planck {tension(t.loc[name], planck):4.1f}σ   vs SH0ES {tension(t.loc[name], shoes):4.1f}σ")

グループごとに重み付き平均もとってみます。

> 注意: 近傍宇宙の測定の多くは同じ超新星のサンプルや同じ較正を共有していて、完全には独立ではありません。
> CMB の 3 つも同じ空を見ています。ここでの平均は、相関を無視したおおまかな目安です。

In [ ]:
def weighted_mean(df):
    w = 1 / df["err"] ** 2
    return np.sum(w * df["H0"]) / np.sum(w), 1 / np.sqrt(np.sum(w))


groups = {}
for g, df in H0_TABLE.groupby("group"):
    mean, err = weighted_mean(df)
    chi2 = np.sum(((df["H0"] - mean) / df["err"]) ** 2)
    groups[g] = dict(H0=mean, err=err)
    print(f"{g:5s}: H0 = {mean:.2f} ± {err:.2f}  (χ² = {chi2:.1f}, {len(df)} 件)")
print(f"early と late の食い違い: {tension(groups['early'], groups['late']):.1f}σ")

近傍宇宙の中では、セファイドを使う SH0ES(73)と、赤色巨星分枝の先端(TRGB)を使う CCHP(70)の間にも少し差があり、これも議論が続いています。
それでも、近傍宇宙の測定はどれも初期宇宙の値より高めに出ています。

## 3. CMB を使わずに初期宇宙側の $H_0$ を出す: BAO + BBN

BAO から決まるのは $H_0 r_d$ と $\Omega_m$ でした([hz_comparison.ipynb](hz_comparison.ipynb) 2 章)。
物差し $r_d$ は、音波が伝わった速さと時間で決まり、主にバリオンの密度 $\omega_b = \Omega_b h^2$ と、物質の密度 $\omega_m = \Omega_m h^2$ に依存します($h = H_0 / 100$)。

数値計算をよく再現する近似式があります(Brieden et al. 2023, DESI 2024 で使われたもの):

$$
r_d \approx 147.05\ \mathrm{Mpc} \left(\frac{\omega_b}{0.02236}\right)^{-0.13} \left(\frac{\omega_{cb}}{0.1432}\right)^{-0.23} \left(\frac{N_{\mathrm{eff}}}{3.04}\right)^{-0.1}
$$

$\omega_{cb}$ は暗黒物質とバリオンの密度で、ニュートリノの分($\omega_\nu \approx 0.0006$)を $\omega_m$ から引いたものです。

**バリオンの密度 $\omega_b$** は、ビッグバンから数分の間にできた重水素の量で決まるので、遠方のクエーサーの光で測った重水素/水素比から分かります(BBN)。
ここでは $\omega_b = 0.02218 \pm 0.00055$(Schöneberg 2024)を使います。

$r_d$ が $h$ に依存するので、「$H_0 \times r_d(h)$ = BAO の値」となる $h$ を数値的に解きます。

In [ ]:
bao = pd.read_csv(BAO_URL + "desi_gaussian_bao_ALL_GCcomb_mean.txt", sep=r"\s+", comment="#", header=None,
                  names=["z", "value", "quantity"])
cov_bao_inv = np.linalg.inv(np.loadtxt(BAO_URL + "desi_gaussian_bao_ALL_GCcomb_cov.txt"))
y_bao = bao["value"].to_numpy()
Z_GRID = np.linspace(0, 2.5, 5001)


def e_lcdm(z, om):
    return np.sqrt(om * (1 + z) ** 3 + 1 - om)


def bao_shape(om):
    # H0 × (DM, DH, DV) / c を BAO の各測定について返す(hz_comparison.ipynb と同じ)
    e = e_lcdm(Z_GRID, om)
    dc = np.concatenate([[0], np.cumsum((1 / e[1:] + 1 / e[:-1]) / 2 * np.diff(Z_GRID))])
    z = bao["z"].to_numpy()
    dm = np.interp(z, Z_GRID, dc)
    dh = 1 / e_lcdm(z, om)
    dv = (z * dm ** 2 * dh) ** (1 / 3)
    q = bao["quantity"].to_numpy()
    return np.select([q == "DM_over_rs", q == "DH_over_rs", q == "DV_over_rs"], [dm, dh, dv])


# (Ωm, H0 rd) の格子で χ²
om_ax = np.linspace(0.26, 0.34, 161)
h0rd_ax = np.linspace(9800, 10500, 281)  # [km/s]
c2 = np.empty((len(om_ax), len(h0rd_ax)))
for i, om in enumerate(om_ax):
    pred = (C_KMS / h0rd_ax[:, None]) * bao_shape(om)[None, :]
    r = y_bao[None, :] - pred
    c2[i] = np.einsum("ij,jk,ik->i", r, cov_bao_inv, r)
p = np.exp(-(c2 - c2.min()) / 2)
p /= p.sum()

# 事後分布から (Ωm, H0 rd) を引く
rng = np.random.default_rng(1)
idx = rng.choice(p.size, size=20000, p=p.ravel())
om_s = om_ax[idx // len(h0rd_ax)] + rng.uniform(-0.5, 0.5, idx.size) * (om_ax[1] - om_ax[0])
h0rd_s = h0rd_ax[idx % len(h0rd_ax)] + rng.uniform(-0.5, 0.5, idx.size) * (h0rd_ax[1] - h0rd_ax[0])
print(f"BAO: Ωm = {om_s.mean():.4f} ± {om_s.std():.4f}, H0 rd = {h0rd_s.mean() / 100:.2f} ± {h0rd_s.std() / 100:.2f} × 100 km/s")

In [ ]:
OMEGA_B_BBN, OMEGA_B_BBN_ERR = 0.02218, 0.00055  # Schöneberg (2024)
OMEGA_NU = 0.00064  # ニュートリノ質量の和 0.06 eV
N_EFF = 3.044


def r_drag(omega_b, omega_cb, n_eff=N_EFF):
    return 147.05 * (omega_b / 0.02236) ** -0.13 * (omega_cb / 0.1432) ** -0.23 * (n_eff / 3.04) ** -0.1


def solve_h0(h0rd, om, omega_b, n_iter=30):
    # H0 × rd(ωb, Ωm h²) = h0rd を、反復で解く
    h0 = np.full_like(np.asarray(h0rd, dtype=float), 70.0)
    for _ in range(n_iter):
        h = h0 / 100
        h0 = h0rd / r_drag(omega_b, om * h ** 2 - OMEGA_NU)
    return h0


# 確認: Planck の値なら rd ≈ 147.1 Mpc になるはず
print(f"Planck の値での rd: {r_drag(0.02237, 0.1200 + 0.02237):.2f} Mpc")

ob_s = rng.normal(OMEGA_B_BBN, OMEGA_B_BBN_ERR, om_s.size)
h0_bbn = solve_h0(h0rd_s, om_s, ob_s)
rd_bbn = h0rd_s / h0_bbn
print(f"BAO + BBN: H0 = {h0_bbn.mean():.2f} ± {h0_bbn.std():.2f} km/s/Mpc, rd = {rd_bbn.mean():.1f} ± {rd_bbn.std():.1f} Mpc")
print(f"DESI の論文の値: 68.51 ± 0.58")

plt.figure(figsize=(7, 4))
plt.hist(h0_bbn, bins=60, density=True, alpha=0.6, label="this notebook: DESI BAO + BBN")
for row, color in [(planck, "tab:blue"), (shoes, "tab:red")]:
    plt.axvspan(row["H0"] - row["err"], row["H0"] + row["err"], color=color, alpha=0.2, label=row.name)
plt.xlabel("H0 [km/s/Mpc]")
plt.legend()
plt.show()

CMB の温度ゆらぎを一切使わなくても、**「初期宇宙の物理で決まる物差し」を使うと $H_0 \approx 68.5$** になります。
ハッブル・テンションは「CMB の解析が間違っている」だけでは説明できず、物差し $r_d$ そのもの(=初期宇宙の物理)か、距離はしごのどちらかに原因があることになります。

## 4. どこが何 % ずれていれば食い違いが消えるか

### (a) 物差し $r_d$ が違っていたら

BAO の $H_0 r_d$ を変えずに $H_0 = 73$ にするには、$r_d$ はいくつである必要があるでしょうか。

In [ ]:
rd_needed = h0rd_s.mean() / shoes["H0"]
print(f"SH0ES の H0 と合うのに必要な rd = {rd_needed:.1f} Mpc(標準の値より {100 * (1 - rd_needed / rd_bbn.mean()):.1f}% 小さい)")

$r_d$ を 6% ほど小さくするには、光子とバリオンが離れる前の宇宙の膨張を少し速くすればよく、
「初期ダークエネルギー」(宇宙の初期に一時的に現れて消えるエネルギー成分)や、未知の軽い粒子(余分な $N_{\mathrm{eff}}$)などの案があります。
ただし、そうすると CMB のほかの特徴とも合わせなければならず、決定打はまだありません。

### (b) 距離はしごの土台(LMC までの距離)が違っていたら

距離はしごの土台の距離指数が $\Delta\mu$ だけ大きい(=遠い)と、セファイドの本当の明るさが明るくなり、超新星の $M_B$ も明るくなり、遠くの超新星までの距離が長くなって $H_0$ は小さくなります。

$$
H_0 \propto 10^{-\Delta\mu / 5}
$$

[distance/cepheid.ipynb](../distance/cepheid.ipynb) で自分で求めた LMC の距離指数を下に入れると、
「もし SH0ES の土台が LMC だけで、LMC の距離が自分の値だったら」の $H_0$ が分かります(SH0ES は LMC の距離指数として食連星の 18.477 を使っています)。

In [ ]:
MU_LMC_DEB = 18.477  # 食連星による LMC の距離指数(Pietrzyński et al. 2019)
MU_LMC_DEB_ERR = 0.026
MU_LMC_MINE = 18.477  # ← cepheid.ipynb で求めた値に書き換える


def h0_with_anchor(mu_lmc):
    return shoes["H0"] * 10 ** (-(mu_lmc - MU_LMC_DEB) / 5)


print(f"自分の LMC 距離指数 {MU_LMC_MINE:.3f} を使った H0 ≈ {h0_with_anchor(MU_LMC_MINE):.1f} km/s/Mpc")

d_mu_needed = 5 * np.log10(shoes["H0"] / planck["H0"])
mu_needed = MU_LMC_DEB + d_mu_needed
print(f"Planck の H0 と合うのに必要な LMC の距離指数: {mu_needed:.3f}"
      f"(距離 {10 ** (mu_needed / 5 + 1) / 1000:.1f} kpc。食連星の値 {10 ** (MU_LMC_DEB / 5 + 1) / 1000:.1f} kpc より"
      f" {100 * (10 ** (d_mu_needed / 5) - 1):.0f}% 遠い = {d_mu_needed / MU_LMC_DEB_ERR:.0f}σ のずれ)")

mu = np.linspace(18.2, 18.8, 100)
plt.figure(figsize=(7, 4))
plt.plot(mu, h0_with_anchor(mu), c="tab:red")
plt.axvspan(MU_LMC_DEB - MU_LMC_DEB_ERR, MU_LMC_DEB + MU_LMC_DEB_ERR, color="gray", alpha=0.3, label="LMC (eclipsing binaries)")
plt.axvline(MU_LMC_MINE, c="k", ls=":", label="my LMC distance (cepheid.ipynb)")
plt.axhspan(planck["H0"] - planck["err"], planck["H0"] + planck["err"], color="tab:blue", alpha=0.2, label="Planck H0")
plt.xlabel("LMC distance modulus μ_LMC [mag]")
plt.ylabel("H0 implied by the ladder [km/s/Mpc]")
plt.legend()
plt.show()

LMC の距離を 8% も間違えるのは、食連星の測定(1% の精度)からは考えにくいことです。
しかも SH0ES は、LMC のほかに天の川銀河のセファイドの視差と、NGC 4258 のメーザーの 3 つを土台にしていて、どれを使っても $H_0$ は 72〜74 になります。
土台の距離だけでテンションを解消するのは難しそうです。

距離はしごの側で疑われているのは、遠くの銀河の中のセファイドの明るさの測り方(混み合った星の光が混ざる)や、超新星の母銀河の違いの影響などで、JWST による検証が進んでいます(CCHP の TRGB との差もその 1 つです)。

## 5. 近くの宇宙だけ速く膨張している?

もし私たちが、まわりより物質の少ない大きな領域(ボイド)の中にいると、その中は少し速く膨張するので、近くだけで測った $H_0$ が大きく出ます。
それなら、**遠くの超新星ほど $H_0$ が小さく** 出るはずです。

[distance/sn_ia_hubble.ipynb](../distance/sn_ia_hubble.ipynb) と同じ方法で、超新星を赤方偏移の範囲で分けて $H_0$ を出してみます。
$z = 0.15$ より遠くまで使うので、近似式ではなく平坦 ΛCDM($\Omega_m = 0.333$)の光度距離を使います。

In [ ]:
raw = pd.read_csv(SN_DATA_URL, sep=r"\s+")
sn = raw.groupby("CID").agg(
    zHD=("zHD", "mean"),
    zHEL=("zHEL", "mean"),
    m_b=("m_b_corr", "mean"),
    m_b_err=("m_b_corr_err_DIAG", "mean"),
    is_cal=("IS_CALIBRATOR", "max"),
    ceph_mu=("CEPH_DIST", "mean"),
).reset_index()

cal = sn[sn["is_cal"] == 1]
w = 1 / cal["m_b_err"] ** 2
M_B = np.sum(w * (cal["m_b"] - cal["ceph_mu"])) / np.sum(w)
print(f"M_B = {M_B:.3f}(較正用 {len(cal)} 個)")

OM_FIT = 0.333
zg = np.linspace(0, 1.0, 4001)
e = e_lcdm(zg, OM_FIT)
dc_hat = np.concatenate([[0], np.cumsum((1 / e[1:] + 1 / e[:-1]) / 2 * np.diff(zg))])  # H0 D_C / c

flow = sn[(sn["is_cal"] == 0) & (sn["zHD"] > 0.01) & (sn["zHD"] < 1.0)].copy()
flow["dL_mpc"] = 10 ** ((flow["m_b"] - M_B - 25) / 5)
flow["H0"] = C_KMS * np.interp(flow["zHD"], zg, dc_hat) * (1 + flow["zHEL"]) / flow["dL_mpc"]

EDGES = [0.01, 0.023, 0.05, 0.1, 0.15, 0.3, 0.5, 1.0]
rows = []
for lo, hi in zip(EDGES[:-1], EDGES[1:]):
    b = flow[(flow["zHD"] >= lo) & (flow["zHD"] < hi)]
    wb = 1 / b["m_b_err"] ** 2
    log_h = np.sum(wb * np.log10(b["H0"])) / np.sum(wb)
    # 誤差: 超新星 1 個の明るさのばらつきから
    scatter = np.std(np.log10(b["H0"]))
    rows.append(dict(z_lo=lo, z_hi=hi, n=len(b), z_mid=b["zHD"].median(), H0=10 ** log_h,
                     err=10 ** log_h * np.log(10) * scatter / np.sqrt(len(b))))
bins = pd.DataFrame(rows)
bins

In [ ]:
plt.figure(figsize=(7, 4))
plt.errorbar(bins["z_mid"], bins["H0"], yerr=bins["err"],
             xerr=[bins["z_mid"] - bins["z_lo"], bins["z_hi"] - bins["z_mid"]], fmt="o", c="tab:red", capsize=3)
plt.axhspan(planck["H0"] - planck["err"], planck["H0"] + planck["err"], color="tab:blue", alpha=0.2, label="Planck")
plt.axhspan(shoes["H0"] - shoes["err"], shoes["H0"] + shoes["err"], color="tab:red", alpha=0.15, label="SH0ES")
plt.xscale("log")
plt.xlabel("redshift range of SNe used")
plt.ylabel("H0 [km/s/Mpc]")
plt.ylim(62, 80)
plt.legend()
plt.show()

どの範囲の超新星を使っても $H_0 \approx 73$ で、遠くで Planck の値に近づく様子はありません($z < 0.023$ は固有運動の影響で少しばらつきます)。
図の誤差棒には入れていませんが、$M_B$ の較正誤差(約 1.4%)が全部の点に共通にかかります。それを考えても、
**近くだけが特別に速く膨張しているという説明は難しい** ことが分かります(必要なボイドは、銀河の分布の観測から許されるよりずっと深く大きいものになります)。

> 注: 高い赤方偏移の点は $\Omega_m$ の値に少し依存します。`OM_FIT` を 0.30 にすると、$z > 0.3$ の点は 1〜2% 動きます。

## まとめ

- 近傍宇宙の測定(70〜74)と、初期宇宙の物差しを使う測定(67〜69)の差は、Planck と SH0ES で約 5σ、グループの平均どうしで約 6σ
- CMB を使わない BAO + BBN でも $H_0 \approx 68.5$ になり、「初期宇宙の物差し」側の値は手法によらずそろっている
- テンションを $r_d$ で解くには約 6% 小さい $r_d$ が、LMC の距離で解くには約 8% 遠い LMC が必要
- 超新星を赤方偏移で分けても $H_0$ は変わらず、近くだけ特別に速く膨張しているわけではなさそう

## 発展メモ

- $N_{\mathrm{eff}}$ を 3.044 から増やすと、BAO + BBN の $H_0$ はどれだけ上がるか(`N_EFF` を変える)
- BBN の $\omega_b$ の誤差を 2 倍にすると、$H_0$ の誤差はどうなるか
- 表に新しい測定(JWST のセファイド、JAGB、重力波の新しいイベントなど)を追加する

## 参考・クレジット

- Planck Collaboration (2020), *Planck 2018 results. VI. Cosmological parameters*, A&A 641, A6
- Aiola et al. (2020), *The Atacama Cosmology Telescope: DR4 Maps and Cosmological Parameters*, JCAP 12, 047
- Balkenhol et al. (2023), *Measurement of the CMB temperature power spectrum and constraints on cosmology from the SPT-3G 2018 TT, TE, and EE dataset*, PRD 108, 023510
- DESI Collaboration (2025), *DESI DR2 Results II: Measurements of Baryon Acoustic Oscillations and Cosmological Constraints*, arXiv:2503.14738
- Riess et al. (2022), *A Comprehensive Measurement of the Local Value of the Hubble Constant ...*, ApJL 934, L7
- Freedman (2021), *Measurements of the Hubble Constant: Tensions in Perspective*, ApJ 919, 16
- Freedman et al. (2025), *Status Report on the Chicago-Carnegie Hubble Program (CCHP): Measurement of the Hubble Constant Using the Hubble and James Webb Space Telescopes*, ApJ 985, 203
- Blakeslee et al. (2021), *The Hubble Constant from Infrared Surface Brightness Fluctuation Distances*, ApJ 911, 65
- Pesce et al. (2020), *The Megamaser Cosmology Project. XIII. Combined Hubble Constant Constraints*, ApJL 891, L1
- Wong et al. (2020), *H0LiCOW XIII. A 2.4% measurement of H0 from lensed quasars*, MNRAS 498, 1420
- TDCOSMO Collaboration (2025), *TDCOSMO 2025: Cosmological constraints from strong lensing time delays*, arXiv:2506.03023
- Abbott et al. (2017), *A gravitational-wave standard siren measurement of the Hubble constant*, Nature 551, 85
- Brieden, Gil-Marín & Verde (2023), *A tale of two (or more) h's*, JCAP 04, 023($r_d$ の近似式)
- Schöneberg (2024), *The 2024 BBN baryon abundance update*, JCAP 06, 006
- Pietrzyński et al. (2019), *A distance to the Large Magellanic Cloud that is precise to one per cent*, Nature 567, 200